In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix,accuracy_score,precision_score,recall_score,roc_curve,auc,roc_auc_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder,RobustScaler,OneHotEncoder
from sklearn.model_selection import cross_val_score,RandomizedSearchCV
from imblearn.over_sampling import ADASYN
from imblearn.pipeline import Pipeline

In [2]:
data = pd.read_csv('Model_Data.csv')
data.drop(['Unnamed: 0','Group','Age_Category', 'Last Contact Day Of The Month',
       'Last Contact Month Of The Year'],axis=1,inplace = True)
data_new=data.copy()
data_new

,Age,Job,Marital Status,Educational Qualification,Contact Communication type,Contact Duration (Seconds),No Of Contacts Performed During This Campaign,Outcome Of Previous Marketing Campaign,Subscription Status
0,58,management,married,tertiary,unknown,261,1,unknown,no
1,44,technician,single,secondary,unknown,151,1,unknown,no
2,33,entrepreneur,married,secondary,unknown,76,1,unknown,no
3,47,blue-collar,married,primary,unknown,92,1,unknown,no
4,33,unknown,single,unknown,unknown,198,1,unknown,no
...,...,...,...,...,...,...,...,...,...
45154,51,technician,married,secondary,cellular,977,3,unknown,yes
45155,71,retired,divorced,primary,cellular,456,2,unknown,yes
45156,72,retired,married,secondary,cellular,1127,5,success,yes
45157,57,blue-collar,married,secondary,telephone,508,4,unknown,no


In [3]:
data_new.columns

Index(['Age', 'Job', 'Marital Status', 'Educational Qualification',
       'Contact Communication type', 'Contact Duration (Seconds)',
       'No Of Contacts Performed During This Campaign',
       'Outcome Of Previous Marketing Campaign', 'Subscription Status'],
      dtype='object')

In [4]:
x=data_new.drop(['Subscription Status'],axis=1)
y = data_new['Subscription Status']

In [5]:
xtrain,xtest,ytrain,ytest=train_test_split(x,y,test_size=0.3,random_state=42)

In [6]:

cat_cols = ['Job', 'Marital Status', 'Educational Qualification' ,'Contact Communication type','Outcome Of Previous Marketing Campaign']

# Outlier numeric
robust_cols = [
    'Age',
    'Contact Duration (Seconds)',
    'No Of Contacts Performed During This Campaign'
]

preprocessor = ColumnTransformer([
    ('num', RobustScaler(), robust_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])




In [7]:
pipeline_adasyn = Pipeline([
    ('preprocessing', preprocessor),
    ('adasyn', ADASYN(random_state=42)),
    ('model', LogisticRegression(max_iter=1000))
])

In [8]:
pipeline_class_weight = Pipeline([
    ('preprocessing', preprocessor),
    ('model', LogisticRegression(class_weight='balanced', max_iter=1000))
])


In [9]:
param_grid_lr = {
    'model__C': [0.01, 0.1, 1, 10]
}

In [10]:
grid_adasyn = RandomizedSearchCV(
    pipeline_adasyn,
    n_iter=5,
    param_distributions=param_grid_lr,
    scoring='f1',
    cv=3,
    n_jobs=-1,
    random_state=42
)

In [11]:
grid_class_weight = RandomizedSearchCV(
    pipeline_class_weight,
    n_iter=5,
    param_distributions=param_grid_lr,
    scoring='f1',
    cv=3,
    n_jobs=-1,
    random_state=42
)


In [12]:
grid_adasyn.fit(xtrain, ytrain)
grid_class_weight.fit(xtrain, ytrain)

RandomizedSearchCV(cv=3,
                   estimator=Pipeline(steps=[('preprocessing',
                                              ColumnTransformer(transformers=[('num',
                                                                               RobustScaler(),
                                                                               ['Age',
                                                                                'Contact '
                                                                                'Duration '
                                                                                '(Seconds)',
                                                                                'No '
                                                                                'Of '
                                                                                'Contacts '
                                                                                'Performed '
                                                                                'During '
                                                                                'This '
                                                                                'Campaign']),
                                                                              ('cat',
                                                                               OneHotEncoder(handle_unknown='ignore'),
                                                                               ['Job',
                                                                                'Marital '
                                                                                'Status',
                                                                                'Educational '
                                                                                'Qualification',
                                                                                'Contact '
                                                                                'Communication '
                                                                                'type',
                                                                                'Outcome '
                                                                                'Of '
                                                                                'Previous '
                                                                                'Marketing '
                                                                                'Campaign'])])),
                                             ('model',
                                              LogisticRegression(class_weight='balanced',
                                                                 max_iter=1000))]),
                   n_iter=5, n_jobs=-1,
                   param_distributions={'model__C': [0.01, 0.1, 1, 10]},
                   random_state=42, scoring='f1')

In [13]:
def evaluate_model(model, X_test, y_test, name):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    print(f"\n{name}")
    print("="*40)
    print("Best Params:", model.best_params_)
    print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("\nClassification Report:\n", classification_report(y_test, y_pred))
    print("ROC-AUC Score:", roc_auc_score(y_test, y_prob))

In [14]:
evaluate_model(grid_adasyn, xtest, ytest, "ADASYN Model")


ADASYN Model
Best Params: {'model__C': 0.01}

Confusion Matrix:
 [[9887 2080]
 [ 224 1357]]

Classification Report:
               precision    recall  f1-score   support

          no       0.98      0.83      0.90     11967
         yes       0.39      0.86      0.54      1581

    accuracy                           0.83     13548
   macro avg       0.69      0.84      0.72     13548
weighted avg       0.91      0.83      0.85     13548

ROC-AUC Score: 0.9068932289919986


In [15]:
evaluate_model(grid_class_weight, xtest, ytest, "Class Weight Model")


Class Weight Model
Best Params: {'model__C': 0.01}

Confusion Matrix:
 [[10278  1689]
 [  273  1308]]

Classification Report:
               precision    recall  f1-score   support

          no       0.97      0.86      0.91     11967
         yes       0.44      0.83      0.57      1581

    accuracy                           0.86     13548
   macro avg       0.71      0.84      0.74     13548
weighted avg       0.91      0.86      0.87     13548

ROC-AUC Score: 0.9079364732034813
